# 流れの図（Flow view）: 遺伝子名を入れると、上流と下流の図ができる

**使い方**: 最初のコードセルの `GENE` に遺伝子名を入れて、上から順に実行する。どんな遺伝子でも、同じ手順で、同じ形の図と表ができる。

## 考え方

### 末端は決めつけない
- これまでの図は、上流の終点を受容体・リガンド、下流の終点を転写因子（TF）と決めていた。この図では**終点を決めつけない**。
- **末端 = これ以上先へ進めない遺伝子**（選んだ基準の辺がもう無い遺伝子）、または最大ステップに着いた遺伝子。受容体・リガンド・TF のこともあれば、キナーゼやそのほかの遺伝子のこともある。
- 受容体・リガンド・TF は、**選ぶ基準には使わない**。点の色で区別し、**図では端に置く**（下の「表示方法」）。

### 使う数字（どの数字を何に使うか）
| 数字 | 中身 | 使い道 |
|---|---|---|
| **辺の証拠 `E`** | OmniPath の論文数（対数）と SIGNOR のスコアを、ノイジーOR で合成した 0〜1 | 経路スコアの元。辺の太さ |
| **`geo`** | √(`weight` × `E`)（`weight` = データベース内の順位ベースのスコア） | 辺を通すかどうかの基準（`GEO_MIN` 以上） |
| **経路スコア `ps`** | 入力遺伝子からその遺伝子までの**最良経路の辺の `E` の積** | 末端側の遺伝子を選ぶ（`N_END`）。経路の強さ |
| **流れ `flow`** | その遺伝子の先（上流なら外側、下流なら外側）にある**末端の遺伝子すべての `ps` の合計** | **途中の遺伝子を選ぶ唯一の基準**（`N_MID`） |
| 末端の数 `n_end` | `flow` に寄与する末端の遺伝子の数 | 流れの内訳（表に出す） |
| DepMap の相関 `r` | 入力遺伝子との遺伝子効果の相関 | **点の大きさだけ**（選ぶ基準には使わない） |
| 機能 | 役割（リガンド・受容体・TF）と UniProt のキーワード（キナーゼ・ホスファターゼ・ユビキチン経路） | **点の色だけ** |

### 選び方（手順）
1. 入力遺伝子から、`geo ≥ GEO_MIN` の辺を、**上流・下流それぞれ**1ホップずつ（最大 `MAX_HOPS`）たどる。前のホップまでに着いた遺伝子は除く。各遺伝子の「親」は、前の層からの辺のうち `E` が最大のもの（最良経路の木）。
2. 木の各遺伝子の `ps` と、`flow`（先の末端の `ps` の合計）を計算する。
3. **`flow` の大きい順に `N_MID` 個**を選ぶ（上流・下流それぞれ）。遺伝子の `flow` は、その先の遺伝子の `flow` 以上なので、選んだ遺伝子は**必ず入力遺伝子までつながる**。
4. さらに、選んだ遺伝子の隣にある（親が選ばれている）遺伝子のうち、**`ps` の大きい順に `N_END` 個**を足す。これで、証拠の強い末端も見える。

### 表示方法
- 中央に入力遺伝子、左に上流、右に下流。列は入力遺伝子からのステップ数（hop）。遺伝子名は点の上部。
- **端の列**: 上流のリガンドは**左端**、受容体はその右の列、下流の TF は**右端**に置く（どれも、その先にさらに別の遺伝子が続かないものだけ。続くものは hop の列のまま、色だけで区別する）。
- **点**: 大きさ = DepMap の相関の大きさ、縁は全部同じ、色 = 機能。
- **辺**: 曲線。**濃い青の実線 = 促進、赤の破線 = 抑制、灰色の点線 = 符号不明**。太さ = 証拠 `E`（0.6 以上は太、0.3〜0.6 は中、0.3 未満は細）。
- 1 つの列に `WRAP_AT`（20）個以上あるときは、半分ずつ二列にする。上流と下流の両方に出る遺伝子は、`E` × `weight` が大きい側に 1 回だけ描く。

### 限界
- 流れは「先の末端の `ps` の合計」なので、末端が多いハブ（ユビキチン経路など）が大きく出ることがある。表の `n_end` と、末端の平均の `ps` で内訳を確かめる。
- ネットワークは、ライセンス判定を通った SIGNOR と OmniPath の符号つきの辺だけ（`network_development.ipynb` の Step 3）。辺の記録が少ない関係（脂質を介する PI3K → AKT1 など）は、`E` が低く出る。

## 1. パラメータ
- 動作: 入力遺伝子と、選び方の数字を決める
- 出力: なし（設定のみ）

In [ ]:
# パラメータ（最初のセルに集約）
GENE = "AKT1"                    # ここに遺伝子名を入れる（HGNC 記号、大文字）
GEO_MIN = 0.25                   # 辺を通す基準: √(weight × E) がこの値以上
MAX_HOPS = 4                     # 入力遺伝子から何ステップまでたどるか（6 にすると、深い遺伝子の影響が増え、探索が大きくなる）
N_MID = 40                       # 上流・下流のそれぞれで、流れの大きい順に描く遺伝子の数
N_END = 15                       # 上流・下流のそれぞれで、さらに描く末端側の遺伝子の数（親が選ばれている遺伝子のうち、経路スコアが大きい順）
MAX_OUT = 3                      # 各遺伝子から描く辺の最大数（最良経路の辺は別に必ず描く）
WRAP_AT = 20                     # 1 つの列に遺伝子がこの数以上あると、半分ずつ二列にする
ROW_PX = 48                      # 1 行の高さ（ピクセル）
CONFIG = "../target_net.yaml"    # DepMap の場所を読む YAML
EDGES_CSV, GENES_CSV = "flow_view_edges.csv", "flow_view_genes.csv"   # network_development.ipynb の Step 26 で作った表

## 2. 表の読み込み
- 動作: 辺の表（`E`・`geo` つき）と遺伝子の表（機能・局在）を読み、入力遺伝子がネットワークにあるか確かめる
- 出力: 辺と遺伝子の数、入力遺伝子から出る辺・入る辺の数

In [ ]:
# 辺と遺伝子の表を読む
import os, math, yaml
import numpy as np, pandas as pd, plotly.graph_objects as go
from collections import defaultdict

edges = pd.read_csv(EDGES_CSV)
genes = pd.read_csv(GENES_CSV, keep_default_na=False).set_index("gene")
if GENE not in set(edges.src) | set(edges.dst):
    raise SystemExit(f"{GENE} はネットワークにない（遺伝子名の綴りか、別名でないか確かめる）")
ok_edges = edges[edges.geo >= GEO_MIN]
print(f"辺 {len(edges)}（geo >= {GEO_MIN}: {len(ok_edges)}）| 遺伝子 {len(genes)} | {GENE} から出る辺 {int((ok_edges.src == GENE).sum())}、入る辺 {int((ok_edges.dst == GENE).sum())}（geo >= {GEO_MIN}）| 機能: {genes.loc[GENE, 'func']}")

## 3. DepMap の相関（点の大きさ用）
- 動作: DepMap の CRISPR 遺伝子効果で、入力遺伝子とネットワークの全遺伝子の相関（ピアソン `r`）を計算する
- 出力: 計算した遺伝子の数。**選ぶ基準には使わない**（点の大きさだけ）

In [ ]:
# 入力遺伝子との DepMap の相関（点の大きさに使う）
eff_f = os.path.join(os.path.expanduser(yaml.safe_load(open(CONFIG))["databases"]["depmap"]["dir"]), "CRISPRGeneEffect.csv")
hdr = pd.read_csv(eff_f, nrows=0).columns
colmap = {c.split(" (")[0].upper(): c for c in hdr if " (" in c}
if GENE in colmap:
    use = (set(edges.src) | set(edges.dst)) & set(colmap) | {GENE}
    dm = pd.read_csv(eff_f, usecols=[colmap[g] for g in use], dtype=np.float32)
    dm.columns = [c.split(" (")[0].upper() for c in dm.columns]                                # usecols は列をファイルの順に返すので、名前は読んだ列から付ける
    corr = dm.corrwith(dm[GENE]).drop(GENE)
    print(f"相関を計算した遺伝子: {len(corr)}（細胞株 {len(dm)}）")
else:
    corr = pd.Series(dtype=float); print(f"{GENE} は DepMap にない → 点の大きさは全部最小")

## 4. 上流・下流へ1ホップずつたどる
- 動作: `geo ≥ GEO_MIN` の辺を、入力遺伝子から上流（辺を逆向きに）・下流（辺の向きに）へ、最大 `MAX_HOPS` ステップたどる。前のホップまでに着いた遺伝子は除く。各遺伝子の親は、前の層からの辺のうち `E` が最大のもの。**遺伝子の種類で止めない**（受容体・TF も先へ進む）
- **上流と下流の両方に出る遺伝子**は、親への辺のスコア **`E` × `weight`** が大きい側だけに置く（1 回目の探索で、両側に出る遺伝子を見つけて、`E` × `weight` が大きい側を決める。2 回目の探索で、もう一方の側から外してやり直す。同じ値なら上流）。そのため、同じ遺伝子が図に 2 回出ることはない
- 出力: 両側に出た遺伝子の数と例、上流・下流の、ホップごとの遺伝子数

In [ ]:
# 上流・下流の探索（遺伝子の種類で止めない）。両側に出る遺伝子は、辺のスコア E × weight が大きい側だけに置く（探索を 2 回行う）
inc = {"down": dict(list(ok_edges.groupby("src"))), "up": dict(list(ok_edges.groupby("dst")))}
blocked = {"up": set(), "down": set()}                                                      # 探索から外す遺伝子（2 回目に使う）
for pass_ in (1, 2):
    tree = {}
    for side in ("up", "down"):
        far = "src" if side == "up" else "dst"                                              # 入力遺伝子から見て遠い側の遺伝子の列
        reached, rec, front = {GENE: 0}, [], [GENE]
        for h in range(1, MAX_HOPS + 1):
            new = []
            for v in front:
                if v not in inc[side]: continue
                g_ = inc[side][v]
                for r in g_[~g_[far].isin(reached) & ~g_[far].isin(blocked[side])].itertuples():
                    other = getattr(r, far)
                    rec.append(dict(hop=h, gene=other, parent=v, effect=r.effect, E=r.E, weight=r.weight, geo=r.geo, mechanism=r.mechanism))
                    if other not in new: new.append(other)
            for g in new: reached[g] = h
            front = new
        tree[side] = pd.DataFrame(rec).sort_values("E").groupby("gene").tail(1).set_index("gene")      # 各遺伝子で E が最大の辺を親にする
    if pass_ == 1:                                                                          # 両側に出る遺伝子は、親への辺の E × weight が大きい側に置く
        both = sorted(set(tree["up"].index) & set(tree["down"].index))
        s_up = tree["up"].loc[both, "E"] * tree["up"].loc[both, "weight"]; s_down = tree["down"].loc[both, "E"] * tree["down"].loc[both, "weight"]
        to_up = {g for g in both if s_up[g] >= s_down[g]}
        blocked["down"], blocked["up"] = to_up, set(both) - to_up                           # 上流に置く遺伝子は下流の探索から外す（逆も同じ）
        assign = pd.DataFrame({"s_up": s_up.round(3), "s_down": s_down.round(3), "side": ["up" if g in to_up else "down" for g in both]}, index=both)
print(f"両側に出た遺伝子: {len(both)} 個 → 上流に置く {len(to_up)}、下流に置く {len(both) - len(to_up)}")
print("例（E × weight が大きい側）: " + ", ".join(f"{g}→{assign.side[g]} ({assign.s_up[g]:.2f} / {assign.s_down[g]:.2f})" for g in ['SRC', 'MTOR', 'GSK3B', 'MAPK1', 'PIK3CA', 'CDK2', 'PTK2'] if g in assign.index))
print(pd.DataFrame({s: tree[s].groupby("hop").size() for s in tree}).fillna(0).astype(int).rename_axis("hop"))

## 5. 経路スコアと流れ
- 動作: 木の各遺伝子の経路スコア `ps`（入力遺伝子までの最良経路の `E` の積）と、流れ `flow`（先の末端の遺伝子すべての `ps` の合計）、末端の数 `n_end` を計算する。末端は、先に子がない遺伝子
- 出力: 末端の遺伝子の機能の内訳（受容体・リガンド・TF 以外が、どれだけあるか）と、流れの上位

In [ ]:
# 経路スコア ps、流れ flow、末端の数 n_end
for side in ("up", "down"):
    T = tree[side]
    ps = {GENE: 1.0}
    for g in T.sort_values("hop").index: ps[g] = T.E[g] * ps[T.parent[g]]                    # 経路スコア = E の積
    kids = defaultdict(list)
    for g in T.index: kids[T.parent[g]].append(g)
    flow, nend = {}, {}
    for g in T.sort_values("hop", ascending=False).index:                                    # 外側（hop が大きい）から内側へ
        ch = kids.get(g, [])
        flow[g] = ps[g] if not ch else sum(flow[c] for c in ch)                              # 末端は自分の ps、そうでなければ子の流れの合計
        nend[g] = 1 if not ch else sum(nend[c] for c in ch)
    T["ps"] = pd.Series(ps); T["flow"] = pd.Series(flow); T["n_end"] = pd.Series(nend); T["is_end"] = [g not in kids for g in T.index]
    T["func"] = genes.func.reindex(T.index).fillna("other")
    ends = T[T.is_end]
    print(f"[{side}] 遺伝子 {len(T)} | 末端 {len(ends)} 個 → 機能の内訳: {ends.func.value_counts().to_dict()}")
    print(f"        流れの上位: " + ", ".join(f"{g} ({T.flow[g]:.2f}, 末端 {int(T.n_end[g])})" for g in T.flow.sort_values(ascending=False).index[:8]))

## 6. 描く遺伝子を選ぶ
- 動作: 上流・下流それぞれで、流れの大きい順に `N_MID` 個を選び、さらに、その隣（親が選ばれている）で経路スコアの大きい順に `N_END` 個を足す
- 出力: 選んだ遺伝子の数

In [ ]:
# 描く遺伝子を選ぶ（流れの大きい順 N_MID 個 + 隣で経路スコアの大きい N_END 個）
sel = {}
for side in ("up", "down"):
    T = tree[side]
    mid = list(T.flow.sort_values(ascending=False).index[:N_MID])
    base_set = set(mid) | {GENE}
    ext = sorted([g for g in T.index if g not in base_set and T.parent[g] in base_set], key=lambda g: -T.ps[g])[:N_END]
    sel[side] = mid + ext
    print(f"[{side}] 描く遺伝子 {len(sel[side])}（流れで {len(mid)} + 隣の末端側で {len(ext)}）")

## 7. 図を描く
- 動作: 左から 上流の hop（遠い順）| 入力遺伝子 | 下流の hop の順に並べ、辺を曲線で描く。点の大きさ = DepMap の相関の大きさ、縁は同じ、色 = 機能。辺の色・線の種類 = 符号、太さ = 証拠 `E`。名前は点の上部。`WRAP_AT` 個以上の列は二列にする
- 出力: 図（画面と HTML・PNG に保存）

In [ ]:
# 図を描く（Plotly）
FUNC_COL = {"ligand": "#79b9a8", "receptor": "#5d84b8", "transcription factor": "#e0b84f", "kinase": "#d98c5f", "phosphatase": "#9b7cb6", "ubiquitin pathway": "#a7855a", "other": "#c4c4c4"}
KINDS = {1: ("activation", "#1f3a8a", "solid"), -1: ("inhibition", "#c0504d", "dash"), 0: ("unknown sign", "#8a8a8a", "dot")}
key = lambda g, side: GENE if g == GENE else f"{g}@{side[0]}"                               # 点のキー（上流は @u、下流は @d。両側に出る遺伝子は別の点になる）
base = lambda k: k.split("@")[0]

func_of = lambda g: genes.func.get(g, "other")
kids_sel = {side: defaultdict(set) for side in tree}                                        # 描く遺伝子のうち、各遺伝子の子（親がその遺伝子のもの）
for side in tree:
    for g in sel[side]: kids_sel[side][tree[side].parent[g]].add(g)
edge_col = {}                                                                               # 端の列に置く遺伝子（先に遺伝子が続くかどうかに関わらず、種類で決める）: 上流のリガンド・受容体は左端、下流の TF は右端
for g in sel["up"]:
    if func_of(g) == "ligand": edge_col[(g, "up")] = "ligands"
    elif func_of(g) == "receptor": edge_col[(g, "up")] = "receptors"
for g in sel["down"]:
    if func_of(g) == "transcription factor": edge_col[(g, "down")] = "transcription factors"
for side in tree:                                                                           # 端に置いた遺伝子の先は描かない（端 = フローの終点）。例外: 受容体の先のリガンド（左端の列）
    T = tree[side]; drop = set()
    for g in sel[side]:
        p = T.parent[g]
        while p != GENE:
            if (p, side) in edge_col and not (side == "up" and edge_col.get((g, side)) == "ligands" and edge_col[(p, side)] == "receptors"): drop.add(g); break
            p = T.parent[p]
    sel[side] = [g for g in sel[side] if g not in drop]
    for g in drop: edge_col.pop((g, side), None)
hop_cols = lambda side: sorted({tree[side].hop[g] for g in sel[side] if (g, side) not in edge_col}, reverse=(side == "up"))
cols = [("ligands", [key(g, "up") for g in sel["up"] if edge_col.get((g, "up")) == "ligands"]), ("receptors", [key(g, "up") for g in sel["up"] if edge_col.get((g, "up")) == "receptors"])]
cols += [(f"up hop {h}", [key(g, "up") for g in sel["up"] if (g, "up") not in edge_col and tree["up"].hop[g] == h]) for h in hop_cols("up")]
cols += [("seed", [GENE])]
cols += [(f"down hop {h}", [key(g, "down") for g in sel["down"] if (g, "down") not in edge_col and tree["down"].hop[g] == h]) for h in hop_cols("down")]
cols += [("transcription factors", [key(g, "down") for g in sel["down"] if edge_col.get((g, "down")) == "transcription factors"])]
cols = [(n, ks) for n, ks in cols if ks]
x = {k: ci for ci, (n, ks) in enumerate(cols) for k in ks}
seed_ci = x[GENE]

edges_d = {}                                                                                # (a, b) -> (E, 符号)。a, b は点のキー。図では左から右へ向かう
for side in ("up", "down"):
    T, S = tree[side], set(sel[side])
    for g in S:                                                                             # 最良経路の辺
        p = T.parent[g]
        a, b = (key(g, side), key(p, side)) if side == "up" else (key(p, side), key(g, side))
        if x[a] < x[b]: edges_d[(a, b)] = (T.E[g], T.effect[g])
    ex = ok_edges[ok_edges.src.isin(S | {GENE}) & ok_edges.dst.isin(S | {GENE}) & (ok_edges.src != ok_edges.dst)]
    ex = ex[[x.get(key(s, side), 9e9) < x.get(key(d, side), -9e9) for s, d in zip(ex.src, ex.dst)]]            # 左から右へ進む辺だけ
    for r in ex.sort_values("E", ascending=False).groupby("src").head(MAX_OUT).itertuples():
        edges_d.setdefault((key(r.src, side), key(r.dst, side)), (r.E, r.effect))

y = {GENE: 0.0}
for ci in range(seed_ci - 1, -1, -1):                                                       # 上流側: 入力遺伝子に近い列から外へ
    ks = cols[ci][1]
    for i, k in enumerate(sorted(ks, key=lambda k: (np.mean([y[b] for (a, b) in edges_d if a == k and b in y] or [1e9]), k))): y[k] = (len(ks) - 1) / 2 - i
for ci in range(seed_ci + 1, len(cols)):                                                    # 下流側
    ks = cols[ci][1]
    for i, k in enumerate(sorted(ks, key=lambda k: (np.mean([y[a] for (a, b) in edges_d if b == k and a in y] or [1e9]), k))): y[k] = (len(ks) - 1) / 2 - i
for ci, (n, ks) in enumerate(cols):                                                         # WRAP_AT 個以上の列は、半分ずつ二列にする
    if len(ks) < WRAP_AT: continue
    order = sorted(ks, key=lambda k: -y[k]); half = (len(order) + 1) // 2
    for part, off in ((order[:half], -0.3), (order[half:], 0.3)):
        for i, k in enumerate(part): x[k] = ci + off; y[k] = (len(part) - 1) / 2 - i

rows_needed = lambda n: (n + 1) // 2 if n >= WRAP_AT else n
ncol, nmax = len(cols), max(rows_needed(len(ks)) for n, ks in cols)
W, Hpx = max(1500, 175 * ncol), max(700, ROW_PX * nmax + 240)
xr, yr = (-0.4, ncol - 0.2), (-nmax / 2 - 1.5, nmax / 2 + 2.5)
sx, sy = (W - 40) / (xr[1] - xr[0]), (Hpx - 140) / (yr[1] - yr[0])

fig = go.Figure()
first_leg = set()
for e_, (kname, kcol, kdash) in KINDS.items():
    for wcond, w_ in ((lambda e: e >= 0.6, 1.8), (lambda e: 0.3 <= e < 0.6, 1.1), (lambda e: e < 0.3, 0.6)):
        xs, ys, hx, hy, hang = [], [], [], [], []
        for (a, b), (E_, ef) in edges_d.items():
            if ef != e_ or not wcond(E_): continue
            (x0, y0), (x1, y1) = (x[a], y[a]), (x[b], y[b]); dx = x1 - x0
            P = np.array([[x0, y0], [x0 + 0.5 * dx, y0], [x1 - 0.5 * dx, y1], [x1, y1]])
            pt = lambda t: (1 - t) ** 3 * P[0] + 3 * (1 - t) ** 2 * t * P[1] + 3 * (1 - t) * t ** 2 * P[2] + t ** 3 * P[3]
            dv = lambda t: 3 * (1 - t) ** 2 * (P[1] - P[0]) + 6 * (1 - t) * t * (P[2] - P[1]) + 3 * t ** 2 * (P[3] - P[2])
            pts = np.array([pt(t) for t in np.linspace(0, 1, 28)]); xs += list(pts[:, 0]) + [None]; ys += list(pts[:, 1]) + [None]
            te = 0.93 if e_ == -1 else 0.88; p_, v_ = pt(te), dv(te); th = math.degrees(math.atan2(v_[0] * sx, v_[1] * sy))
            hx.append(p_[0]); hy.append(p_[1]); hang.append(th - 90 if e_ == -1 else th)
        if not xs: continue
        showleg = kname not in first_leg; first_leg.add(kname)
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=kcol, width=w_, dash=kdash), opacity=0.75, hoverinfo="skip", name=f"{kname} edge", legendgroup=kname, showlegend=showleg))
        fig.add_trace(go.Scatter(x=hx, y=hy, mode="markers", hoverinfo="skip", showlegend=False, legendgroup=kname,
                                 marker=dict(symbol="line-ns" if e_ == -1 else "arrow", size=7 + 2 * w_, angle=hang, angleref="up", color=kcol, line=dict(color=kcol, width=2.5) if e_ == -1 else None)))
allk = [k for k in x if k != GENE]
for cat in FUNC_COL:
    ks = [k for k in allk if genes.func.get(base(k), "other") == cat]
    if not ks: continue
    r_ = [corr.get(base(k), np.nan) for k in ks]
    sz = [8 + 70 * (abs(v) if v == v else 0) for v in r_]
    fig.add_trace(go.Scatter(x=[x[k] for k in ks], y=[y[k] for k in ks], mode="markers", marker=dict(size=sz, color=FUNC_COL[cat], line=dict(color="#444444", width=1.5)), name=cat,
                             hovertext=[f"<b>{base(k)}</b><br>{cat}<br>DepMap r with {GENE}: {('%.2f' % v) if v == v else 'n/a'}" for k, v in zip(ks, r_)], hoverinfo="text"))
    for k, s in zip(ks, sz):                                                                # 名前は自分の点の上部（点の上端に少し重なる位置）
        fig.add_annotation(x=x[k], y=y[k], text=base(k), showarrow=False, xanchor="center", yanchor="bottom", yshift=max(s / 2 - 3, 2), font=dict(size=11, color="#222222"))
for v in (0.1, 0.2, 0.4):
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=8 + 70 * v, color="#ffffff", line=dict(color="#444444", width=1.5)), name=f"|r| = {v}"))
for lab, w_ in (("evidence E >= 0.6 (thick)", 1.8), ("0.3 <= E < 0.6", 1.1), ("E < 0.3 (thin)", 0.6)):
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="#666666", width=w_), name=lab))
fig.add_trace(go.Scatter(x=[x[GENE]], y=[0], mode="markers+text", text=[GENE], textposition="top center", textfont=dict(size=15), name="seed",
                         marker=dict(size=22, color="#555555", symbol="diamond"), hoverinfo="skip"))
for ci, (n, ks) in enumerate(cols): fig.add_annotation(x=ci, y=yr[1] - 0.6, text=n, showarrow=False, font=dict(size=13, color="#666666"))
fig.update_layout(width=W, height=Hpx, plot_bgcolor="white", margin=dict(l=20, r=20, t=30, b=170), xaxis=dict(visible=False, range=list(xr)), yaxis=dict(visible=False, range=list(yr)),
                  legend=dict(orientation="h", yanchor="top", y=-0.01, xanchor="center", x=0.5, font=dict(size=11)))
fig.write_html(f"flow_view_{GENE}.html", include_plotlyjs="cdn")
try: fig.write_image(f"flow_view_{GENE}.png", scale=1.4)
except Exception as e: print("PNG は保存できなかった:", str(e)[:80])
print(f"描いた点 {len(x) - 1} 個 | 辺 {len(edges_d)} 本")
fig.show()

## 8. 遺伝子の表を保存する
- 動作: 上流・下流の全遺伝子について、経路・符号・経路スコア・流れ・末端の数・機能・DepMap の相関・図に描いたかを表にして CSV に保存する
- 出力: 流れの大きい順の上位 30 遺伝子の表、CSV ファイル（`flow_view_<遺伝子名>_genes.csv`）

In [ ]:
# 遺伝子の表
rows = []
for side in ("up", "down"):
    T = tree[side]
    for g in T.index:
        p, sg, q = [g], T.effect[g], T.parent[g]
        while q != GENE: p.append(q); sg *= T.effect[q]; q = T.parent[q]
        path = " > ".join(p + [GENE]) if side == "up" else " > ".join([GENE] + p[::-1])
        rows.append(dict(gene=g, side=side, hop=int(T.hop[g]), parent=T.parent[g], path=path, path_sign={1: "activation", -1: "inhibition", 0: "unknown"}[sg],
                         path_score=round(T.ps[g], 4), flow=round(T.flow[g], 4), n_end=int(T.n_end[g]), is_end=bool(T.is_end[g]), E_to_parent=round(T.E[g], 3), geo_to_parent=round(T.geo[g], 3),
                         mechanism=T.mechanism[g], function=T.func[g], locations=genes.locations.get(g, ""), uniprot_flags=genes.uniprot_flags.get(g, ""),
                         depmap_r=round(corr.get(g, np.nan), 3), drawn=g in set(sel[side]), in_both_sides_before=g in assign.index, s_up=assign.s_up.get(g, np.nan), s_down=assign.s_down.get(g, np.nan)))
tab = pd.DataFrame(rows).sort_values(["flow", "path_score"], ascending=False).reset_index(drop=True)
tab.to_csv(f"flow_view_{GENE}_genes.csv", index=False, encoding="utf-8-sig")
print(f"保存: flow_view_{GENE}_genes.csv | {len(tab)} 行（遺伝子 × 側）")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 60)
tab[["gene", "side", "hop", "parent", "path_sign", "path_score", "flow", "n_end", "is_end", "function", "depmap_r", "drawn"]].head(30)